# MixMIL with AnnData: binary classification

This is the single-phenotype version of the donor-level workflow. Each donor is a bag of cells and the binary phenotype is repeated in `.obs` for the cells belonging to that donor.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import torch

from mixmil import MixMIL
from mixmil.anndata import add_attention_to_anndata, prepare_anndata

rng = np.random.default_rng(11)
n_bags, cells_per_bag, n_features = 60, 25, 10
embedding = rng.normal(size=(n_bags * cells_per_bag, n_features)).astype('float32')
bag_embedding = embedding.reshape(n_bags, cells_per_bag, n_features).mean(axis=1)
phenotype_logit = bag_embedding[:, 0] - 0.75 * bag_embedding[:, 1] + rng.normal(scale=0.4, size=n_bags)
phenotype = (phenotype_logit > 0).astype('float32')
obs = pd.DataFrame({
    'donor': np.repeat([f'donor_{i:03d}' for i in range(n_bags)], cells_per_bag),
    'split': np.repeat(np.where(np.arange(n_bags) < 45, 'train', 'test'), cells_per_bag),
    'case_control': np.repeat(phenotype, cells_per_bag),
})
adata = ad.AnnData(X=np.zeros((len(obs), 1), dtype='float32'), obs=obs)
adata.obsm['X_mixmil'] = embedding
adata

In [ ]:
prepared = prepare_anndata(
    adata,
    bag_key='donor',
    target_key='case_control',
    feature_key='X_mixmil',
    split_key='split',
)

model = MixMIL(
    Q=prepared.n_features,
    K=prepared.n_fixed_effects,
    P=1,
    likelihood='binomial',
    n_trials=1,
    mean_field=True,
)
model.fit(prepared.train.Xs, prepared.train.F, prepared.train.Y, n_epochs=150, batch_size=16, verbose=False)
logits = model.predict_logits(prepared.test.Xs, prepared.test.F)
probabilities = torch.sigmoid(logits).squeeze(1)
predicted_labels = (probabilities >= 0.5).to(torch.int64)
print('first five probabilities:', probabilities[:5])
print('first five labels:', predicted_labels[:5])

The returned `prepared.test.bag_ids` gives the exact donor order corresponding to the prediction rows.

In [ ]:
result = add_attention_to_anndata(
    adata, prepared.test, model.get_weights(prepared.test.Xs)[0],
    key='mixmil_attention',
)
result.uns['mixmil_binary'] = {
    'bag_ids': prepared.test.bag_ids,
    'logits': logits.detach().cpu().numpy().ravel().tolist(),
    'probabilities': probabilities.detach().cpu().numpy().tolist(),
}
result.obs[['donor', 'mixmil_attention']].head()